# Infinity-2B GGUF Figure 10 quantitative-style sample

This notebook uses the unofficial Q8_0 Infinity-2B GGUF port from Hugging Face. It is a paper-faithful adaptation to Infinity's GGUF inference path, not an exact reproduction of the paper's 1M/1024px FP16 setup.

For evaluation-sized test runs, it uses only the 10 Figure 10 styles marked for quantitative evaluation and randomly samples 3 Parti prompts for each style:

```text
10 styles x 3 prompts = 30 cases per generation step
```

The intended flow is:

```text
Infinity-2B baseline
PFB + SAC
Multi-step PFB + SAC with decay at steps 2, 4, 6, 8
Top-1 SVD PFB + SAC with style-related steps
Final aggregate comparison
Download outputs
```

The notebook is inference-only and image-focused. It does not calculate metrics.


In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import time

# All files are kept under one directory so the notebook can be rerun safely.
ROOT = Path('/content/notebook_09_infinity2b_gguf')
PORT_DIR = ROOT / 'gguf_port'
OFFICIAL_DIR = PORT_DIR / 'Infinity'
ASSET_DIR = ROOT / 'assets'
OUTPUT_DIR = ROOT / 'outputs'
for path in (PORT_DIR, ASSET_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'
MODEL_PN = '0.25M'  # Colab-friendly 512px adaptation; the paper reports 1M/1024px.
CFG_SCALE = 1.0
TAU = 0.1
SEED = 42
T5_DEVICE = 'cuda'  # T4/Colab: keep T5 off host RAM; use 'cpu' only with a high-RAM runtime.
print('ROOT:', ROOT)
print('MODEL_PN:', MODEL_PN, '| CFG:', CFG_SCALE, '| TAU:', TAU, '| SEED:', SEED, '| T5:', T5_DEVICE)

In [ ]:
# Check the runtime before installing anything.
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 2**30, 2))
else:
    print('WARNING: no GPU detected; inference will be extremely slow.')

In [ ]:
# Install the packages used by the GGUF loader.
# We intentionally do not install torch or flash-attn here: Colab already ships torch,
# and the GGUF loader falls back to PyTorch SDPA when flash-attn is unavailable.
packages = [
    'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia',
    'gputil', 'colorama', 'omegaconf', 'timm==0.9.6',
    'decord', 'pytz', 'imageio', 'einops', 'opencv-python', 'accelerate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependency installation finished.')

In [ ]:
# Clone the official Python architecture only once.
if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

# Download only the files needed from the unofficial GGUF repository.
from huggingface_hub import hf_hub_download

def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(
        repo_id=GGUF_REPO,
        filename=filename,
        local_dir=str(target_dir),
    ))

PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
PATCH_DIR = ROOT / 'gguf_patched_source'
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', PATCH_DIR)
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', PATCH_DIR)

# The GGUF repository includes patched source files with optional attention fallbacks.
# Copy them over the matching files in the official source tree.
official_basic = OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py'
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_BASIC, official_basic)
shutil.copy2(PATCHED_INFINITY, official_infinity)

# The patched attention module may intentionally expose flash_attn_func=None.
# Guard the official constructor so it selects the PyTorch SDPA fallback safely.
infinity_source = official_infinity.read_text()
old_attention_guard = "customized_kernel_installed = any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
new_attention_guard = "customized_kernel_installed = flash_attn_func is not None and any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
if old_attention_guard not in infinity_source:
    raise RuntimeError('Expected optional-attention guard was not found in patched infinity.py')
official_infinity.write_text(infinity_source.replace(old_attention_guard, new_attention_guard, 1))
INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

print('GGUF model:', INFINITY_GGUF)
print('T5 encoder:', T5_GGUF)
print('VAE:', VAE_PATH)
print('Loader:', PORT_SCRIPT)
print('Loader utility:', PORT_UTILS)

In [ ]:
# Verify the expected files before importing the custom loader.
required_files = [PORT_SCRIPT, PORT_UTILS, PATCHED_BASIC, PATCHED_INFINITY, INFINITY_GGUF, T5_GGUF, VAE_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

for path in required_files:
    print(f'{path.name:40s} {path.stat().st_size / 2**30:.3f} GiB')

assert (OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py').exists(), 'Official Infinity source is incomplete.'
print('All GGUF, VAE, and official source files are present.')

## Memory-efficient T5 loading

The upstream GGUF script first materializes a complete FP16 state dictionary and then creates a second T5 model. That duplicates most of the encoder in host RAM. The replacement below streams one GGUF tensor at a time into an empty Flan-T5-XL model. It preserves the expected 2048-dimensional text features.

A smaller T5 (such as T5-small/base) is not a drop-in replacement: its hidden width and learned feature space differ from the Infinity-2B cross-attention interface. Using one would require a trained projection or distillation step, so this notebook keeps Flan-T5-XL and reduces peak RAM instead.

In [ ]:
import math
import numpy as np
import gguf

def load_t5_encoder_streaming(gguf_path, device='cpu'):
    from gguf import GGUFReader
    from transformers import T5Config, T5EncoderModel

    key_map = {
        'enc.': 'encoder.',
        '.blk.': '.block.',
        'token_embd': 'shared',
        'output_norm': 'final_layer_norm',
        'attn_q': 'layer.0.SelfAttention.q',
        'attn_k': 'layer.0.SelfAttention.k',
        'attn_v': 'layer.0.SelfAttention.v',
        'attn_o': 'layer.0.SelfAttention.o',
        'attn_norm': 'layer.0.layer_norm',
        'attn_rel_b': 'layer.0.SelfAttention.relative_attention_bias',
        'ffn_up': 'layer.1.DenseReluDense.wi_1',
        'ffn_down': 'layer.1.DenseReluDense.wo',
        'ffn_gate': 'layer.1.DenseReluDense.wi_0',
        'ffn_norm': 'layer.1.layer_norm',
    }

    config = T5Config.from_pretrained('google/flan-t5-xl')
    try:
        from accelerate import init_empty_weights
        with init_empty_weights():
            model = T5EncoderModel(config)
        # Materialize directly as FP16 to avoid allocating a full FP32 T5.
        model = model.to(dtype=torch.float16)
        model.to_empty(device=device)
    except Exception as exc:
        raise RuntimeError(
            'Streaming T5 loading requires the accelerate package and empty-weight support. '
            'Restart the runtime and rerun the dependency cell.'
        ) from exc

    model.eval()
    model.requires_grad_(False)
    parameter_refs = dict(model.named_parameters())
    buffer_refs = dict(model.named_buffers())
    reader = GGUFReader(str(gguf_path))
    quantized_types = {gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16}
    loaded = 0
    skipped = []

    print(f'[Streaming T5 load] {gguf_path} -> {device}')
    with torch.inference_mode():
        for tensor in reader.tensors:
            name = tensor.name
            for old_key, new_key in key_map.items():
                name = name.replace(old_key, new_key)
            shape = torch.Size(tuple(int(v) for v in reversed(tensor.shape)))
            raw = torch.from_numpy(np.array(tensor.data))
            is_quantized = tensor.tensor_type not in quantized_types
            if is_quantized:
                quant_param = gguf_loader.GGUFParameter(raw, quant_type=tensor.tensor_type)
                value = gguf_loader.dequantize_gguf_tensor(quant_param, target_dtype=torch.float16)
            else:
                value = raw.to(dtype=torch.float16)
            if value.numel() != math.prod(shape):
                skipped.append((name, 'numel mismatch'))
                del raw, value
                continue
            value = value.reshape(shape)
            target = parameter_refs.get(name)
            if target is None:
                target = buffer_refs.get(name)
            if target is None or tuple(target.shape) != tuple(shape):
                skipped.append((name, 'missing or shape mismatch'))
                del raw, value
                continue
            target.data.copy_(value.to(device=target.device, dtype=target.dtype))
            loaded += 1
            del raw, value

    del reader, parameter_refs, buffer_refs
    gc.collect()
    # The model was materialized on the requested device already.
    # Keep this safety path for unusual device-string inputs.
    if str(next(model.parameters()).device) != str(torch.device(device)):
        model.to(device)
    model.eval()
    model.requires_grad_(False)
    print(f'[Streaming T5 load complete] tensors loaded: {loaded}, skipped: {len(skipped)}')
    if skipped:
        print('First skipped tensors:', skipped[:5])
    return model

print('Memory-efficient T5 loader is ready.')

## Import the unofficial loader

The upstream GGUF script contains a NumPy 2 compatibility assignment to `np.ndarray.newbyteorder`. That assignment can fail on some Colab runtimes because NumPy types are immutable. The next cell creates a temporary sanitized copy of the loader and removes only that obsolete compatibility block.

In [ ]:
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_source = PORT_SCRIPT.read_text()
compat_pattern = r"\n    # Apply NumPy 2\.0 compatibility patch.*?\n    # Load GGUF state dict"
loader_source, replacements = re.subn(
    compat_pattern,
    '\n    # NumPy compatibility is handled by the installed gguf package.\n    # Load GGUF state dict',
    loader_source,
    count=1,
    flags=re.S,
)
print('Removed obsolete NumPy compatibility block:', replacements == 1)

PATCHED_LOADER = PORT_DIR / 'generate_image_2b_q8_gguf_colab.py'
PATCHED_LOADER.write_text(loader_source)
spec = importlib.util.spec_from_file_location('infinity_gguf_colab_loader', PATCHED_LOADER)
gguf_loader = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gguf_loader
spec.loader.exec_module(gguf_loader)
print('Custom GGUF loader imported successfully.')

## Load all components

The T5 encoder is streamed directly to CUDA to reduce Colab host-RAM pressure. The VAE and quantized Infinity transformer are placed on the GPU.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('A CUDA GPU is required for practical inference. Select a GPU runtime and rerun.')

print('[1/4] Loading T5 tokenizer...')
text_tokenizer = gguf_loader.load_t5_tokenizer_from_gguf(str(T5_GGUF))

print(f'[2/4] Streaming quantized T5 encoder to {T5_DEVICE}...')
text_encoder = load_t5_encoder_streaming(str(T5_GGUF), device=T5_DEVICE)

print('[3/4] Loading VAE on GPU...')
vae = gguf_loader.load_vae(str(VAE_PATH), vae_type=32, device=DEVICE)

print('[4/4] Loading quantized Infinity-2B transformer on GPU...')
infinity_model = gguf_loader.load_infinity_from_gguf(
    str(INFINITY_GGUF),
    vae=vae,
    device=DEVICE,
    model_type='infinity_2b',
    text_channels=2048,
    pn=MODEL_PN,
)

infinity_model.eval()
vae.eval()
print('All components loaded successfully.')

In [ ]:
# Build the official dynamic-resolution schedule for the selected preset.
import numpy as np
from infinity.utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates

ASPECT_RATIO = 1.0
h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - ASPECT_RATIO))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template][MODEL_PN]['scales']
scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]
print('Aspect ratio:', h_div_w_template)
print('Preset:', MODEL_PN)
print('Scale schedule:', scale_schedule)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def tensor_to_pil(image):
    """Convert common Infinity output layouts/ranges into an RGB PIL image."""
    if isinstance(image, (list, tuple)):
        image = image[0]
    tensor = image.detach().float().cpu() if torch.is_tensor(image) else torch.as_tensor(image).float()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Unexpected image shape: {tuple(tensor.shape)}')
    if tensor.shape[0] in (1, 3, 4):
        tensor = tensor.permute(1, 2, 0)
    if tensor.shape[-1] == 1:
        tensor = tensor.repeat(1, 1, 3)
    if tensor.shape[-1] > 3:
        tensor = tensor[..., :3]
    lo, hi = float(tensor.min()), float(tensor.max())
    if lo < -0.05:
        tensor = (tensor + 1.0) / 2.0
    elif hi > 1.05:
        tensor = tensor / 255.0
    array = (tensor.clamp(0, 1).numpy() * 255).round().astype('uint8')
    return Image.fromarray(array, mode='RGB')

def generate_one(prompt, seed=SEED, output_path=None):
    started = time.time()
    with torch.inference_mode():
        image = gguf_loader.generate_image(
            infinity_model, vae, text_tokenizer, text_encoder, prompt,
            cfg_scale=CFG_SCALE,
            tau=TAU,
            seed=seed,
            scale_schedule=scale_schedule,
            vae_type=32,
            device=DEVICE,
        )
    pil = tensor_to_pil(image)
    if output_path is not None:
        pil.save(output_path)
    del image
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Generated in {time.time() - started:.2f}s:', prompt)
    return pil

In [ ]:
import types
from contextlib import nullcontext
import torchvision
import torch.nn.functional as F
from tqdm.auto import tqdm
from PIL import Image, ImageOps
from infinity.models.basic import CrossAttnBlock, apply_rotary_emb, slow_attn
from infinity.models.infinity import sample_with_top_k_top_p_also_inplace_modifying_logits_

RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
device = DEVICE
infinity = infinity_model
SCALE_SCHEDULE = scale_schedule
PATCH_NUMS = tuple(h for (_, h, w) in SCALE_SCHEDULE)
IMAGE_SIZE_HW = (512, 512)

if not hasattr(vae.quantizer, 'lfq'):
    vae.quantizer.lfq = vae.quantizer.bsq

print('Infinity-2B compatibility aliases ready.')
print('Backend: Infinity-2B GGUF | device:', device, '| image size:', IMAGE_SIZE_HW)

## 2. Quantitative styles, random prompt sampling, and shared configuration


In [ ]:
import csv
import random

VAR_SOICT_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd().parent.parent,
    RUNTIME_ROOT / 'VAR_SOICT',
    RUNTIME_ROOT / 'Image Generation' / 'VAR_SOICT',
]


def _find_var_soict_root():
    for candidate in VAR_SOICT_CANDIDATES:
        if (
            (candidate / 'styles' / 'quantitative_eval_styles_10.csv').exists()
            and (candidate / 'prompts' / 'content_prompts_190.csv').exists()
        ):
            return candidate.resolve()
    raise FileNotFoundError(
        'Could not find VAR_SOICT. Run this notebook from the VAR_SOICT folder, '
        'or copy the VAR_SOICT folder to /content before running this cell.'
    )


VAR_SOICT_ROOT = _find_var_soict_root()
STYLE_FIGURE10_DIR = VAR_SOICT_ROOT / 'styles'
QUANT_STYLE_CSV = STYLE_FIGURE10_DIR / 'quantitative_eval_styles_10.csv'
CONTENT_PROMPT_CSV = VAR_SOICT_ROOT / 'prompts' / 'content_prompts_190.csv'

OUTPUT_DIR = RUNTIME_ROOT / 'Infinity_outputs' / 'infinity2b_random_3_prompts_per_eval_style'
BASELINE_DIR = OUTPUT_DIR / 'baseline'
VARIANT_DIR = OUTPUT_DIR / 'variants'
AGGREGATE_DIR = OUTPUT_DIR / 'aggregate'
for output_subdir in (BASELINE_DIR, VARIANT_DIR, AGGREGATE_DIR):
    output_subdir.mkdir(parents=True, exist_ok=True)

CFG = 1.0
TOP_K = 600
TOP_P = 0.95
SEED = 42
TAU = 0.1

PROMPTS_PER_STYLE = 3
PROMPT_SAMPLE_SEED = 42
EXPECTED_EVAL_STYLES = 10
EXPECTED_CASES_PER_VARIANT = EXPECTED_EVAL_STYLES * PROMPTS_PER_STYLE

PAPER_ALPHA = 1.0
PAPER_PFB_FEATURE_INDEX = 2       # F3 in the paper; zero-based Infinity stage 2.
# The paper defines S_fine={3,...,S}; with zero-based Infinity indices, SAC
# therefore starts at stage index 2, the same fine stage as PFB.
PAPER_SAC_PREDICTION_START = 2
BASE_STYLE_STRENGTH = 0.8      # PFB and PFB + SAC single-stage interventions.
MULTISCALE_STYLE_STRENGTH = 1.0  # Compensates for decay across later injection stages.
MULTISCALE_FEATURE_INDICES = [2, 4, 6, 8]
MULTISCALE_STYLE_DECAY = 0.75

TOP1_STYLE_FEATURE_INDICES = [0, 1, 2, 9]
TOP1_STYLE_SVD_RANK = 1
TOP1_STYLE_STRENGTH = 1.0
TOP1_STYLE_DECAY = 0.75

RUN_BASELINE_SECTION = True
RUN_PFB_SAC_SECTION = True
RUN_MULTISCALE_SECTION = True
RUN_TOP1_STYLE_STEPS_SECTION = True
RUN_AGGREGATE_SECTION = True


def _read_csv(path):
    with Path(path).open(newline='', encoding='utf-8') as fp:
        return list(csv.DictReader(fp))


def _styled_prompt(content, style):
    descriptor = style['style_descriptor'].strip()
    if descriptor.lower().endswith('style'):
        return f"{content['content_prompt']}, {content['superclass']}, in {descriptor}"
    return f"{content['content_prompt']}, {content['superclass']}, in {descriptor} style"


STYLE_ROWS = _read_csv(QUANT_STYLE_CSV)
CONTENT_ROWS = _read_csv(CONTENT_PROMPT_CSV)

if len(STYLE_ROWS) != EXPECTED_EVAL_STYLES:
    raise RuntimeError(f'Expected {EXPECTED_EVAL_STYLES} quantitative styles, found {len(STYLE_ROWS)}.')
if len(CONTENT_ROWS) != 190:
    raise RuntimeError(f'Expected 190 Parti content prompts, found {len(CONTENT_ROWS)}.')
if len(CONTENT_ROWS) < PROMPTS_PER_STYLE:
    raise RuntimeError(f'Need at least {PROMPTS_PER_STYLE} prompts to sample per style.')

FIGURE10_SESSIONS = []
FIGURE10_CASES = []
case_manifest_rows = []
rng = random.Random(PROMPT_SAMPLE_SEED)

for session_id, style in enumerate(STYLE_ROWS):
    style_path = VAR_SOICT_ROOT / style['style_reference_image']
    if not style_path.exists():
        raise FileNotFoundError(style_path)

    sampled_prompts = rng.sample(CONTENT_ROWS, k=PROMPTS_PER_STYLE)
    session_cases = []
    session = {
        'session_id': session_id,
        'name': f"Figure 10 style {int(style['figure10_index']):02d}: {style['style_name']}",
        'style_path': style_path,
        'style_label': style['style_name'],
        'style_id': style['style_id'],
        'figure10_index': int(style['figure10_index']),
        'prompts': [],
    }

    for prompt_id, content in enumerate(sampled_prompts):
        prompt = _styled_prompt(content, style)
        case = {
            'case_id': f"{style['style_id']}_p{prompt_id + 1:02d}",
            'session_id': session_id,
            'session_name': session['name'],
            'style_path': style_path,
            'style_label': style['style_name'],
            'style_id': style['style_id'],
            'figure10_index': int(style['figure10_index']),
            'content_id': content['content_id'],
            'content_prompt': content['content_prompt'],
            'category': content['category'],
            'superclass': content['superclass'],
            'prompt': prompt,
        }
        session['prompts'].append(prompt)
        session_cases.append(case)
        FIGURE10_CASES.append(case)
        case_manifest_rows.append({
            'case_id': case['case_id'],
            'style_id': case['style_id'],
            'figure10_index': case['figure10_index'],
            'style_name': case['style_label'],
            'content_id': case['content_id'],
            'content_prompt': case['content_prompt'],
            'superclass': case['superclass'],
            'prompt': case['prompt'],
            'style_reference_image': style['style_reference_image'],
        })

    session['cases'] = session_cases
    FIGURE10_SESSIONS.append(session)

if len(FIGURE10_CASES) != EXPECTED_CASES_PER_VARIANT:
    raise RuntimeError(
        f'Expected {EXPECTED_CASES_PER_VARIANT} cases, found {len(FIGURE10_CASES)}.'
    )

SELECTED_CASES_CSV = OUTPUT_DIR / 'selected_cases_30.csv'
with SELECTED_CASES_CSV.open('w', newline='', encoding='utf-8') as fp:
    writer = csv.DictWriter(fp, fieldnames=list(case_manifest_rows[0].keys()))
    writer.writeheader()
    writer.writerows(case_manifest_rows)

print('VAR_SOICT root:', VAR_SOICT_ROOT)
print(f'Quantitative styles: {len(FIGURE10_SESSIONS)}')
print(f'Random prompts per style: {PROMPTS_PER_STYLE}')
print(f'Cases per variant: {len(FIGURE10_CASES)}')
print(f'Selected case manifest: {SELECTED_CASES_CSV}')
print(f'Infinity sampling: CFG={CFG}, top-k={TOP_K}, top-p={TOP_P}, tau={TAU}, seed={SEED}')
print(f'Prompt sampling seed: {PROMPT_SAMPLE_SEED}')
for session in FIGURE10_SESSIONS:
    print(f"- {session['style_id']} | Figure 10 #{session['figure10_index']:02d} | {session['style_label']} | {len(session['prompts'])} prompts")


## 3. Principal Feature Blending (paper Eq. 5-6)


In [ ]:

def load_reference_image(path, size=512):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return image_01.mul(2).sub(1), image_01, image


@torch.no_grad()
def extract_multiscale_style_features(image_m11):
    # Infinity's equivalent of the paper's F_s: cumulative BSQ residual codes
    # at the final latent resolution, one snapshot after each AR scale.
    with torch.amp.autocast('cuda', enabled=False):
        _, _, _, all_bit_indices, _, _ = vae.encode(image_m11.float(), scale_schedule=SCALE_SCHEDULE)

    summed_codes = None
    features = []
    final_size = SCALE_SCHEDULE[-1]
    num_scales = len(SCALE_SCHEDULE)
    for step_id, bit_indices in enumerate(all_bit_indices):
        codes = vae.quantizer.lfq.indices_to_codes(bit_indices, label_type='bit_label')
        if step_id != num_scales - 1:
            codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)
        summed_codes = codes if summed_codes is None else summed_codes + codes
        features.append(summed_codes.detach().float().clone())
    assert len(features) == len(PATCH_NUMS)
    return features


def phi_svd(feature, alpha=1.0, rank=None):
    '''Paper Eq. (5), generalized from VAR [B,C,H,W] to Infinity [B,C,T,H,W].'''
    original_dtype = feature.dtype
    batch, channels = feature.shape[:2]
    spatial_shape = feature.shape[2:]
    outputs = []

    for batch_id in range(batch):
        matrix = feature[batch_id].detach().float().reshape(channels, -1)
        u, singular_values, vh = torch.linalg.svd(matrix, full_matrices=False)
        available_rank = singular_values.numel()
        used_rank = available_rank if rank is None else min(int(rank), available_rank)
        weights = torch.exp(
            -float(alpha) * torch.arange(used_rank, device=matrix.device, dtype=matrix.dtype)
        )
        weighted_s = singular_values[:used_rank] * weights
        reconstructed = (u[:, :used_rank] * weighted_s.unsqueeze(0)) @ vh[:used_rank]
        outputs.append(reconstructed.reshape(channels, *spatial_shape))

    return torch.stack(outputs).to(dtype=original_dtype)


def principal_feature_blend(generation_feature, style_feature, alpha=1.0, rank=None, strength=1.0):
    '''PFB with optional strength multiplier; strength=1 is the paper method.'''
    if generation_feature.shape != style_feature.shape:
        raise ValueError(f'PFB shape mismatch: {generation_feature.shape} vs {style_feature.shape}')
    style_feature = style_feature.to(generation_feature)
    style_component = phi_svd(style_feature, alpha=alpha, rank=rank)
    generation_component = phi_svd(generation_feature, alpha=alpha, rank=rank)
    return generation_feature + float(strength) * (style_component - generation_component)


def apply_feature_edit(generation_feature, style_feature, mode, alpha=1.0, rank=None, strength=1.0):
    if mode == 'none':
        return generation_feature
    if mode == 'replace':
        return style_feature.to(generation_feature)
    if mode == 'pfb':
        return principal_feature_blend(
            generation_feature, style_feature, alpha=alpha, rank=rank, strength=strength
        )
    raise ValueError(f'Unknown edit mode: {mode}')


## 4. Structural Attention Correction (paper Eq. 7)


In [ ]:

class SACController:
    def __init__(self, base_batch=1):
        self.base_batch = base_batch
        self.active = False
        self.sac_strength = 1.0
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0

    def reset_statistics(self):
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0


def _infinity_sac_attention_forward(attention, x, attn_bias_or_two_vector, attn_fn=None, scale_schedule=None, rope2d_freqs_grid=None, scale_ind=0):
    batch4, length, channels = x.shape
    if attention.using_flash:
        raise RuntimeError('Notebook SAC patch expects customized_flash_attn=False.')

    # GGUF replaces mat_qkv with GGUFLinear; its forward dequantizes Byte weights on demand.
    qkv = attention.mat_qkv(x)
    qkv = qkv + torch.cat((attention.q_bias, attention.zero_k_bias, attention.v_bias)).to(qkv)
    qkv = qkv.view(batch4, length, 3, attention.num_heads, attention.head_dim)
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)

    if attention.cos_attn:
        scale_mul = attention.scale_mul_1H11.clamp_max(attention.max_scale_mul).exp()
        q = F.normalize(q, dim=-1, eps=1e-12).mul(scale_mul).contiguous()
        k = F.normalize(k, dim=-1, eps=1e-12).contiguous()
        v = v.contiguous()
    else:
        q, k, v = q.contiguous(), k.contiguous(), v.contiguous()

    if rope2d_freqs_grid is not None:
        q, k = apply_rotary_emb(
            q, k, scale_schedule, rope2d_freqs_grid,
            attention.pad_to_multiplier, attention.rope2d_normalized_by_hw, scale_ind,
        )

    controller = getattr(attention, '_paper_sac_controller', None)
    if controller is not None and controller.active:
        b = controller.base_batch
        if batch4 != 4 * b:
            raise RuntimeError(f'SAC expected joint batch {4 * b}, received {batch4}.')

        # Order: content conditional, generation conditional,
        #        content unconditional, generation unconditional.
        q_content = torch.cat((q[:b], q[:b], q[2*b:3*b], q[2*b:3*b]), dim=0)
        k_content = torch.cat((k[:b], k[:b], k[2*b:3*b], k[2*b:3*b]), dim=0)
        if controller.sac_strength >= 1.0:
            q, k = q_content, k_content
        else:
            q = q + float(controller.sac_strength) * (q_content - q)
            k = k + float(controller.sac_strength) * (k_content - k)

        controller.total_calls += 1
        controller.max_q_copy_error = max(
            controller.max_q_copy_error,
            float((q[b:2*b] - q[:b]).abs().max().detach().cpu()),
            float((q[3*b:4*b] - q[2*b:3*b]).abs().max().detach().cpu()),
        )
        controller.max_k_copy_error = max(
            controller.max_k_copy_error,
            float((k[b:2*b] - k[:b]).abs().max().detach().cpu()),
            float((k[3*b:4*b] - k[2*b:3*b]).abs().max().detach().cpu()),
        )

    if attention.caching:
        if attention.cached_k is None:
            attention.cached_k, attention.cached_v = k, v
        else:
            attention.cached_k = torch.cat((attention.cached_k, k), dim=2)
            attention.cached_v = torch.cat((attention.cached_v, v), dim=2)
        k, v = attention.cached_k, attention.cached_v

    if attention.use_flex_attn and attn_fn is not None:
        output = attn_fn(q, k, v, scale=attention.scale).transpose(1, 2).reshape(batch4, length, channels)
    else:
        output = slow_attn(
            query=q.to(v.dtype), key=k.to(v.dtype), value=v,
            scale=attention.scale, attn_mask=attn_bias_or_two_vector, dropout_p=0,
        ).transpose(1, 2).reshape(batch4, length, channels)
    return attention.proj_drop(attention.proj(output))


class PaperSACPatch:
    def __init__(self, model, controller):
        self.model = model
        self.controller = controller
        self.original_forwards = []

    def __enter__(self):
        for block in self.model.unregistered_blocks:
            if not isinstance(block, CrossAttnBlock):
                continue
            attention = block.sa
            self.original_forwards.append((attention, attention.forward))
            attention._paper_sac_controller = self.controller
            attention.forward = types.MethodType(_infinity_sac_attention_forward, attention)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        for attention, original_forward in self.original_forwards:
            attention.forward = original_forward
            if hasattr(attention, '_paper_sac_controller'):
                delattr(attention, '_paper_sac_controller')
        return False


## 5. Joint dual-path autoregressive inference


In [ ]:

def encode_prompts(prompts, enable_positive_prompt=False):
    if isinstance(prompts, str):
        prompts = [prompts]
    tokens = text_tokenizer(
        text=list(prompts), max_length=512, padding='max_length', truncation=True, return_tensors='pt'
    )
    input_ids = tokens.input_ids.to(device, non_blocking=True)
    mask = tokens.attention_mask.to(device, non_blocking=True)
    with torch.no_grad():
        text_features = text_encoder(input_ids=input_ids, attention_mask=mask)['last_hidden_state'].float()
    lens = mask.sum(dim=-1).tolist()
    cu_seqlens_k = F.pad(mask.sum(dim=-1).to(dtype=torch.int32).cumsum_(0), (1, 0))
    max_seqlen_k = max(lens)
    kv_compact = []
    for len_i, feat_i in zip(lens, text_features.unbind(0)):
        kv_compact.append(feat_i[:len_i])
    kv_compact = torch.cat(kv_compact, dim=0)
    return kv_compact, lens, cu_seqlens_k, max_seqlen_k


def _sample_bit_labels(logits_bl2d, rng, top_k, top_p):
    batch, seq_len = logits_bl2d.shape[:2]
    logits = logits_bl2d.reshape(batch, -1, 2).clone()
    sampled = sample_with_top_k_top_p_also_inplace_modifying_logits_(
        logits, rng=rng, top_k=top_k, top_p=top_p, num_samples=1
    )[:, :, 0]
    return sampled.reshape(batch, seq_len, -1)


def _bit_labels_to_codes(idx_bld, pn):
    idx = idx_bld.reshape(idx_bld.shape[0], pn[1], pn[2], -1)
    idx = idx.unsqueeze(1)  # [B, 1, h, w, d]
    return vae.quantizer.lfq.indices_to_codes(idx, label_type='bit_label')


def _next_raw_from_summed_codes(summed_codes, next_scale):
    last_stage = F.interpolate(summed_codes, size=next_scale, mode=vae.quantizer.z_interplote_up)
    last_stage = last_stage.squeeze(-3)
    if infinity.apply_spatial_patchify:
        last_stage = torch.nn.functional.pixel_unshuffle(last_stage, 2)
    last_stage = last_stage.reshape(*last_stage.shape[:2], -1).permute(0, 2, 1)
    return last_stage


def _decode_summed_codes_to_image_01(summed_codes):
    image = vae.decode(summed_codes.squeeze(-3))
    return image.add(1).mul(0.5).clamp(0, 1)


@torch.no_grad()
def paper_dual_path_generate(
    model,
    prompt,
    style_features,
    *,
    seed=42,
    cfg=1.0,
    tau=0.1,
    top_k=900,
    top_p=0.97,
    pfb_feature_index=2,
    pfb_feature_indices=None,
    sac_prediction_start=3,
    edit_mode='pfb',
    alpha=1.0,
    rank=None,
    style_strength=1.0,
    style_decay=1.0,
    style_strength_by_step=None,
    sac_strength=1.0,
    enable_sac=True,
):
    '''Paper Algorithm 1 adapted to Infinity's bitwise 0.25M GGUF inference.

    The paper writes SAC as active from the fine stage s=3. Here the code uses
    zero-based stage indices, so SAC starts at index 2 and remains active for
    every later stage. The multi-scale PFB configuration is an experiment
    added on top of the paper's single PFB intervention at F3.
    '''
    if cfg < 1.0:
        raise ValueError('CFG must be >= 1.0 for this dual-stream experiment.')
    if pfb_feature_indices is None:
        pfb_feature_indices = [pfb_feature_index]
    else:
        pfb_feature_indices = sorted(set(int(index) for index in pfb_feature_indices))
    if not pfb_feature_indices or any(index < 0 or index >= len(SCALE_SCHEDULE) for index in pfb_feature_indices):
        raise ValueError('Invalid PFB feature indices.')
    if style_strength_by_step is not None:
        style_strength_by_step = {int(step): float(strength) for step, strength in style_strength_by_step.items()}
        expected_steps = set(pfb_feature_indices)
        if set(style_strength_by_step) != expected_steps or any(strength < 0 for strength in style_strength_by_step.values()):
            raise ValueError('style_strength_by_step must provide one non-negative strength for every PFB scale.')
    if enable_sac and not 0 <= sac_prediction_start < len(SCALE_SCHEDULE):
        raise ValueError('Invalid SAC prediction start.')

    model.eval()
    base_batch = 1
    condition_batch = 2  # content stream + generation stream
    content_rng = torch.Generator(device=device).manual_seed(seed)
    generation_rng = torch.Generator(device=device).manual_seed(seed)

    kv_compact, lens, cu_seqlens_k, max_seqlen_k = encode_prompts([prompt, prompt])
    kv_compact_un = kv_compact.clone()
    total = 0
    for le in lens:
        kv_compact_un[total:total + le] = model.cfg_uncond[:le]
        total += le
    kv_compact = torch.cat((kv_compact, kv_compact_un), dim=0)
    cu_seqlens_k = torch.cat((cu_seqlens_k, cu_seqlens_k[1:] + cu_seqlens_k[-1]), dim=0)
    bs = 4

    kv_compact = model.text_norm(kv_compact)
    sos = cond_BD = model.text_proj_for_sos((kv_compact, cu_seqlens_k, max_seqlen_k))
    kv_compact = model.text_proj_for_ca(kv_compact)
    ca_kv = kv_compact, cu_seqlens_k, max_seqlen_k
    last_stage = sos.unsqueeze(1).expand(bs, 1, -1) + model.pos_start.expand(bs, 1, -1)

    with torch.amp.autocast('cuda', enabled=False):
        cond_BD_or_gss = model.shared_ada_lin(cond_BD.float()).float().contiguous()

    final_size = SCALE_SCHEDULE[-1]
    content_summed = last_stage.new_zeros(base_batch, model.d_vae, *final_size)
    generation_summed = torch.zeros_like(content_summed)
    content_trace, generation_trace = [], []

    controller = SACController(base_batch)
    controller.sac_strength = float(sac_strength)
    controller.reset_statistics()

    for block in model.unregistered_blocks:
        block.sa.kv_caching(True)

    pre_pfb_max_difference = 0.0
    pfb_relative_change_by_step = {}
    try:
        with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16, cache_enabled=True):
            sac_context = PaperSACPatch(model, controller) if enable_sac else nullcontext()
            with sac_context:
                for step_id, pn in enumerate(SCALE_SCHEDULE):
                    controller.active = enable_sac and step_id >= sac_prediction_start
                    need_to_pad = 0
                    attn_fn = None
                    if model.use_flex_attn:
                        attn_fn = model.attn_fn_compile_dict.get(tuple(SCALE_SCHEDULE[:step_id + 1]), None)

                    layer_idx = 0
                    for block_idx, block_chunk in enumerate(model.block_chunks):
                        if model.add_lvl_embeding_only_first_block and block_idx == 0:
                            last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                        if not model.add_lvl_embeding_only_first_block:
                            last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)

                        for block in block_chunk.module:
                            last_stage = block(
                                x=last_stage,
                                cond_BD=cond_BD_or_gss,
                                ca_kv=ca_kv,
                                attn_bias_or_two_vector=None,
                                attn_fn=attn_fn,
                                scale_schedule=SCALE_SCHEDULE,
                                rope2d_freqs_grid=model.rope2d_freqs_grid,
                                scale_ind=step_id,
                            )
                            layer_idx += 1

                    logits = model.get_logits(last_stage, cond_BD).mul(1 / float(tau))
                    logits = float(cfg) * logits[:condition_batch] + (1 - float(cfg)) * logits[condition_batch:]
                    content_idx = _sample_bit_labels(logits[:1], content_rng, top_k, top_p)
                    generation_idx = _sample_bit_labels(logits[1:2], generation_rng, top_k, top_p)

                    content_codes = _bit_labels_to_codes(content_idx, pn)
                    generation_codes = _bit_labels_to_codes(generation_idx, pn)
                    if step_id != len(SCALE_SCHEDULE) - 1:
                        content_codes = F.interpolate(content_codes, size=final_size, mode=vae.quantizer.z_interplote_up)
                        generation_codes = F.interpolate(generation_codes, size=final_size, mode=vae.quantizer.z_interplote_up)

                    content_summed = content_summed + content_codes
                    generation_summed = generation_summed + generation_codes

                    if step_id < min(pfb_feature_indices):
                        pre_pfb_max_difference = max(
                            pre_pfb_max_difference,
                            float((content_summed - generation_summed).abs().max().detach().cpu()),
                        )

                    if step_id in pfb_feature_indices and edit_mode != 'none':
                        injection_order = pfb_feature_indices.index(step_id)
                        effective_strength = (
                            style_strength_by_step[step_id]
                            if style_strength_by_step is not None
                            else float(style_strength) * float(style_decay) ** injection_order
                        )
                        generation_before_edit = generation_summed.clone()
                        generation_summed = apply_feature_edit(
                            generation_summed,
                            style_features[step_id],
                            mode=edit_mode,
                            alpha=alpha,
                            rank=rank,
                            strength=effective_strength,
                        )
                        pfb_relative_change_by_step[step_id] = float(
                            (generation_summed - generation_before_edit).norm()
                            / generation_before_edit.norm().clamp_min(1e-8)
                        )

                    content_trace.append(content_summed.detach().float().clone())
                    generation_trace.append(generation_summed.detach().float().clone())

                    if step_id != len(SCALE_SCHEDULE) - 1:
                        next_scale = SCALE_SCHEDULE[step_id + 1]
                        content_next = _next_raw_from_summed_codes(content_summed, next_scale)
                        generation_next = _next_raw_from_summed_codes(generation_summed, next_scale)
                        two_streams = torch.cat((content_next, generation_next), dim=0)
                        last_stage = model.word_embed(model.norm0_ve(two_streams))
                        last_stage = last_stage.repeat(bs // condition_batch, 1, 1)

        content_image = _decode_summed_codes_to_image_01(content_summed)
        generation_image = _decode_summed_codes_to_image_01(generation_summed)
        return {
            'content_image_01': content_image,
            'stylized_image_01': generation_image,
            'content_features': content_trace,
            'generation_features': generation_trace,
            'pre_pfb_max_difference': pre_pfb_max_difference,
            'sac_calls': controller.total_calls,
            'max_q_copy_error': controller.max_q_copy_error,
            'max_k_copy_error': controller.max_k_copy_error,
            'pfb_relative_change_by_step': pfb_relative_change_by_step,
        }
    finally:
        controller.active = False
        for block in model.unregistered_blocks:
            block.sa.kv_caching(False)


## 6. Shared image-only experiment helpers


In [ ]:
def _safe_name(text):
    return ''.join(character if character.isalnum() else '_' for character in str(text)).strip('_').lower()


def _image_tensor_to_pil(image):
    tensor = image.detach().float().cpu()
    if tensor.ndim == 4:
        tensor = tensor[0]
    array = tensor.clamp(0, 1).permute(1, 2, 0).mul(255).byte().numpy()
    return Image.fromarray(array)


def _save_image_tensor(image, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _image_tensor_to_pil(image).save(path)


STYLE_IMAGE_CACHE = {}
STYLE_FEATURE_CACHE = {}


def get_style_image(style_path):
    key = str(Path(style_path))
    if key not in STYLE_IMAGE_CACHE:
        _, style_01, _ = load_reference_image(style_path)
        STYLE_IMAGE_CACHE[key] = style_01.detach().float().cpu()
    return STYLE_IMAGE_CACHE[key]


def get_style_features(style_path):
    key = str(Path(style_path))
    if key not in STYLE_FEATURE_CACHE:
        style_m11, _, _ = load_reference_image(style_path)
        with torch.inference_mode():
            features = extract_multiscale_style_features(style_m11)
        STYLE_FEATURE_CACHE[key] = [feature.detach().float().cpu() for feature in features]
        del style_m11, features
        gc.collect()
        torch.cuda.empty_cache()
    return [feature.to(device) for feature in STYLE_FEATURE_CACHE[key]]


def generate_content_image(prompt):
    with torch.inference_mode():
        result = paper_dual_path_generate(
            infinity,
            prompt,
            [],
            seed=SEED,
            cfg=CFG,
            tau=TAU,
            top_k=TOP_K,
            top_p=TOP_P,
            pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
            sac_prediction_start=PAPER_SAC_PREDICTION_START,
            edit_mode='none',
            enable_sac=False,
        )
    image = result['content_image_01'].detach().float().cpu()
    del result
    gc.collect()
    torch.cuda.empty_cache()
    return image


def generate_variant_image(
    prompt, style_path, *, pfb_feature_indices, style_decay, style_strength, enable_sac, rank=None,
    style_strength_by_step=None,
):
    style_features = get_style_features(style_path)
    with torch.inference_mode():
        result = paper_dual_path_generate(
            infinity,
            prompt,
            style_features,
            seed=SEED,
            cfg=CFG,
            tau=TAU,
            top_k=TOP_K,
            top_p=TOP_P,
            pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
            pfb_feature_indices=pfb_feature_indices,
            sac_prediction_start=PAPER_SAC_PREDICTION_START,
            edit_mode='pfb',
            alpha=PAPER_ALPHA,
            rank=rank,
            style_strength=style_strength,
            style_decay=style_decay,
            style_strength_by_step=style_strength_by_step,
            sac_strength=1.0,
            enable_sac=enable_sac,
        )
    image = result['stylized_image_01'].detach().float().cpu()
    del style_features, result
    gc.collect()
    torch.cuda.empty_cache()
    return image


def _plot_image(axis, image):
    axis.imshow(image[0].detach().float().clamp(0, 1).permute(1, 2, 0).numpy())
    axis.axis('off')


def _show_variant_session(session, case_images, variant_name, save_path):
    import textwrap
    columns = ['style reference'] + [case['prompt'] for case in FIGURE10_CASES if case['session_id'] == session['session_id']]
    figure, axes = plt.subplots(1, len(columns), figsize=(3.2 * len(columns), 4.2), squeeze=False)
    axes = axes[0]
    _plot_image(axes[0], get_style_image(session['style_path']))
    axes[0].set_title(f"style reference\n{session['style_label']}", fontsize=10)
    session_cases = [case for case in FIGURE10_CASES if case['session_id'] == session['session_id']]
    for column_id, case in enumerate(session_cases, start=1):
        _plot_image(axes[column_id], case_images[case['case_id']])
        axes[column_id].set_title(f'"{textwrap.fill(case["prompt"], width=26)}"', fontsize=9)
    figure.suptitle(f'{variant_name} | {session["name"]}', fontsize=14)
    figure.tight_layout()
    figure.savefig(save_path, dpi=180, bbox_inches='tight')
    print('saved:', save_path)
    plt.show()
    plt.close(figure)


def run_variant_section(
    variant_name, *, pfb_feature_indices, style_decay, style_strength, enable_sac, rank=None,
    style_strength_by_step=None,
):
    section_dir = VARIANT_DIR / _safe_name(variant_name)
    section_dir.mkdir(parents=True, exist_ok=True)
    results = {}
    for case in tqdm(FIGURE10_CASES, desc=variant_name):
        image = generate_variant_image(
            case['prompt'],
            case['style_path'],
            pfb_feature_indices=pfb_feature_indices,
            style_decay=style_decay,
            style_strength=style_strength,
            enable_sac=enable_sac,
            rank=rank,
            style_strength_by_step=style_strength_by_step,
        )
        results[case['case_id']] = image
        _save_image_tensor(image, section_dir / f"{case['case_id']}.png")

    for session in FIGURE10_SESSIONS:
        gallery_path = section_dir / f"session_{session['session_id'] + 1:02d}.png"
        _show_variant_session(session, results, variant_name, gallery_path)
    return results


## 7. Step 1: Baseline images from Infinity-2B


In [ ]:
BASELINE_RESULTS = {}

if RUN_BASELINE_SECTION:
    for case in tqdm(FIGURE10_CASES, desc='Infinity-2B baseline content images'):
        image = generate_content_image(case['prompt'])
        BASELINE_RESULTS[case['case_id']] = image
        _save_image_tensor(image, BASELINE_DIR / f"{case['case_id']}.png")

print(f'baseline images available: {len(BASELINE_RESULTS)}')

# Quality gate: inspect every baseline session before interpreting style variants.
# Each gallery contains the style reference followed by all baseline object images.
if BASELINE_RESULTS:
    print('Baseline galleries:')
    for session in FIGURE10_SESSIONS:
        gallery_path = BASELINE_DIR / f"session_{session['session_id'] + 1:02d}.png"
        _show_variant_session(session, BASELINE_RESULTS, 'Infinity-2B baseline', gallery_path)


## 8. Step 2: PFB + SAC

This is the main single-step style-transfer variant: PFB at the paper pivot feature followed by structural attention correction.


In [ ]:
PFB_SAC_RESULTS = {}

if RUN_PFB_SAC_SECTION:
    PFB_SAC_RESULTS = run_variant_section(
        'PFB + SAC',
        pfb_feature_indices=[PAPER_PFB_FEATURE_INDEX],
        style_decay=1.0,
        style_strength=BASE_STYLE_STRENGTH,
        enable_sac=True,
    )

print(f'PFB + SAC images available: {len(PFB_SAC_RESULTS)} / {len(FIGURE10_CASES)}')


## 9. Step 3: Multi-step PFB + SAC with decay at 2, 4, 6, 8

This variant applies PFB repeatedly at zero-based style steps `2, 4, 6, 8`, keeps SAC enabled, and decays style strength geometrically across those steps.


In [ ]:
MULTISCALE_RESULTS = {}

if RUN_MULTISCALE_SECTION:
    MULTISCALE_RESULTS = run_variant_section(
        'Multi-step PFB + SAC (steps 2,4,6,8; strength=1.0, decay=0.75)',
        pfb_feature_indices=MULTISCALE_FEATURE_INDICES,
        style_decay=MULTISCALE_STYLE_DECAY,
        style_strength=MULTISCALE_STYLE_STRENGTH,
        enable_sac=True,
    )

print(f'multi-step decay images available: {len(MULTISCALE_RESULTS)} / {len(FIGURE10_CASES)}')


## 10. Step 4: Top-1 SVD PFB + SAC with style-related steps

This style-related-steps variant keeps SAC enabled, restricts PFB to the top SVD component (`rank=1`), and injects at steps `0, 1, 2, 9`.


In [ ]:
TOP1_STYLE_STEPS_RESULTS = {}

if max(TOP1_STYLE_FEATURE_INDICES) >= len(SCALE_SCHEDULE):
    raise RuntimeError(
        f'This variant requires at least {max(TOP1_STYLE_FEATURE_INDICES) + 1} Infinity scales; '
        f'got {len(SCALE_SCHEDULE)}.'
    )

if RUN_TOP1_STYLE_STEPS_SECTION:
    TOP1_STYLE_STEPS_RESULTS = run_variant_section(
        'Top-1 SVD PFB + SAC (style steps 0,1,2,9; strength=1.0, decay=0.75)',
        pfb_feature_indices=TOP1_STYLE_FEATURE_INDICES,
        style_decay=TOP1_STYLE_DECAY,
        style_strength=TOP1_STYLE_STRENGTH,
        enable_sac=True,
        rank=TOP1_STYLE_SVD_RANK,
    )

print(f'top-1 style-step images available: {len(TOP1_STYLE_STEPS_RESULTS)} / {len(FIGURE10_CASES)}')


## Output layout

The notebook writes results under:

```text
Infinity_outputs/infinity2b_random_3_prompts_per_eval_style/
```

Each generation step saves 30 per-case PNGs: 10 quantitative-eval styles x 3 randomly sampled Parti prompts per style. The selected prompt/style pairs are saved to:

```text
selected_cases_30.csv
```

The final aggregate section produces one row per sampled prompt with this format:

```text
style reference | baseline | PFB + SAC | multi-step decay | top-1 style steps
```

The prompt text is shown below the style reference image for that row.


## 11. Final aggregate comparison

Run this after the baseline and all three style variants finish. Each aggregate sheet uses the requested row layout: style reference, baseline, PFB + SAC, multi-step decay, and top-1 style steps.


In [ ]:
import textwrap


AGGREGATE_COLUMNS = [
    ('style reference', None),
    ('baseline', BASELINE_RESULTS),
    ('PFB + SAC', PFB_SAC_RESULTS),
    ('multi-step decay', MULTISCALE_RESULTS),
    ('top-1 style steps', TOP1_STYLE_STEPS_RESULTS),
]


def _plot_labeled_image(axis, image, label, *, prompt=None):
    _plot_image(axis, image)
    axis.set_title(label, fontsize=11, pad=10)
    if prompt is not None:
        axis.text(
            0.5,
            -0.10,
            f'"{textwrap.fill(prompt, width=36)}"',
            ha='center',
            va='top',
            transform=axis.transAxes,
            fontsize=10,
            clip_on=False,
        )


def _show_aggregate_session(session, save_path):
    session_cases = [case for case in FIGURE10_CASES if case['session_id'] == session['session_id']]
    figure, axes = plt.subplots(
        len(session_cases),
        len(AGGREGATE_COLUMNS),
        figsize=(3.35 * len(AGGREGATE_COLUMNS), 3.75 * len(session_cases)),
        squeeze=False,
    )
    for row_id, case in enumerate(session_cases):
        for column_id, (label, results) in enumerate(AGGREGATE_COLUMNS):
            axis = axes[row_id, column_id]
            if results is None:
                _plot_labeled_image(
                    axis,
                    get_style_image(session['style_path']),
                    label,
                    prompt=case['prompt'],
                )
            else:
                _plot_labeled_image(axis, results[case['case_id']], label)

    figure.suptitle(f'Infinity-2B final aggregate comparison | {session["name"]}', fontsize=14, y=0.995)
    figure.subplots_adjust(left=0.02, right=0.995, top=0.92, bottom=0.08, wspace=0.18, hspace=0.52)
    figure.savefig(save_path, dpi=180, bbox_inches='tight')
    print('saved:', save_path)
    plt.show()
    plt.close(figure)


if RUN_AGGREGATE_SECTION:
    expected = len(FIGURE10_CASES)
    available = {label: len(results) for label, results in AGGREGATE_COLUMNS if results is not None}
    if any(count != expected for count in available.values()):
        raise RuntimeError(
            f'Run baseline and all three style variants before final aggregate comparison: '
            f'{available}, expected {expected} each.'
        )
    for session in FIGURE10_SESSIONS:
        _show_aggregate_session(
            session,
            AGGREGATE_DIR / f"session_{session['session_id'] + 1:02d}_final_aggregate.png",
        )


## Download outputs

Run this final cell in Colab to download the complete experiment output folder as a ZIP file.


In [ ]:
from pathlib import Path
import shutil
from google.colab import files

src = Path('/content/Infinity_outputs/infinity2b_random_3_prompts_per_eval_style')
zip_path = Path('/content/infinity2b_random_3_prompts_per_eval_style.zip')

if not src.exists():
    raise FileNotFoundError(f'Output folder not found: {src}')

shutil.make_archive(
    str(zip_path.with_suffix('')),
    'zip',
    root_dir=src.parent,
    base_dir=src.name,
)
files.download(str(zip_path))
